============================================================
Deep Learning Audio Classification with STT and TTS
CNN + TensorFlow + SpeechRecognition + gTTS
Google Colab notebook
============================================================
This notebook trains a CNN to classify short spoken commands,
evaluates it, and provides speech-to-text / text-to-speech demos.

In [ ]:
# 1. Install required libraries
%pip install -q SpeechRecognition gTTS librosa scikit-learn seaborn

In [ ]:
# 2. Imports and reproducibility
import os
import time
import random
import zipfile
import pathlib
import shutil
import wave
import urllib.request
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import speech_recognition as sr

from gtts import gTTS
from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 3. Download and extract the Mini Speech Commands dataset
DATA_ROOT = pathlib.Path("/content/audio_project_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_ROOT / "mini_speech_commands.zip"
EXTRACT_ROOT = DATA_ROOT / "extracted"
DATASET_PATH = EXTRACT_ROOT / "mini_speech_commands"

URL = (
    "https://storage.googleapis.com/"
    "download.tensorflow.org/data/mini_speech_commands.zip"
)

if not ZIP_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(URL, ZIP_PATH)

print("Extracting dataset...")
if not DATASET_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(EXTRACT_ROOT)

if not DATASET_PATH.exists():
    candidates = list(EXTRACT_ROOT.rglob("mini_speech_commands"))
    if not candidates:
        raise FileNotFoundError("Dataset folder not found after extraction.")
    DATASET_PATH = candidates[0]

print("Dataset path:", DATASET_PATH)

In [ ]:
# 4. Validate dataset WAV files and identify classes
class_dirs = [
    p for p in DATASET_PATH.iterdir()
    if p.is_dir() and not p.name.startswith(".")
]

if not class_dirs:
    raise RuntimeError("No class directories found in the dataset.")

bad_files = []
valid_counts = {}

for folder in class_dirs:
    count = 0
    for file in folder.iterdir():
        if not file.is_file() or file.suffix.lower() != ".wav":
            continue
        try:
            with wave.open(str(file), "rb") as wav:
                if wav.getnframes() <= 0:
                    raise ValueError("Empty WAV file")
            count += 1
        except Exception:
            bad_files.append(file)
    valid_counts[folder.name] = count

for file in bad_files:
    try:
        file.unlink()
    except OSError:
        pass

CLASS_NAMES = sorted(
    p.name for p in DATASET_PATH.iterdir()
    if p.is_dir() and any(p.glob("*.wav"))
)
NUM_CLASSES = len(CLASS_NAMES)

print("Valid WAV counts:", valid_counts)
print("Invalid files removed:", len(bad_files))
print("Classes:", CLASS_NAMES)

if NUM_CLASSES < 2:
    raise RuntimeError("Not enough valid audio classes were found.")

In [ ]:
# 5. Create reproducible train / validation / test splits
records = []

for label_id, class_name in enumerate(CLASS_NAMES):
    files = sorted((DATASET_PATH / class_name).glob("*.wav"))
    files = [str(f) for f in files]
    random.Random(SEED).shuffle(files)

    n = len(files)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    for file_path in files[:n_train]:
        records.append((file_path, label_id, "train"))
    for file_path in files[n_train:n_train + n_val]:
        records.append((file_path, label_id, "val"))
    for file_path in files[n_train + n_val:]:
        records.append((file_path, label_id, "test"))

split_df = pd.DataFrame(records, columns=["path", "label", "split"])

print("Split counts:")
display(split_df.groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:
# 6. Audio preprocessing and TensorFlow input pipelines
SAMPLE_RATE = 16000
AUDIO_LENGTH = 16000
FRAME_LENGTH = 255
FRAME_STEP = 128
FFT_LENGTH = 256
BATCH_SIZE = 64

def load_waveform(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(audio) < AUDIO_LENGTH:
        audio = np.pad(audio, (0, AUDIO_LENGTH - len(audio)))
    else:
        audio = audio[:AUDIO_LENGTH]
    return audio.astype(np.float32)

def make_spectrogram(waveform):
    stft = tf.signal.stft(
        waveform,
        frame_length=FRAME_LENGTH,
        frame_step=FRAME_STEP,
        fft_length=FFT_LENGTH,
    )
    return tf.math.log(tf.abs(stft) + 1e-6)[..., tf.newaxis]

def load_example(path, label):
    audio = tf.numpy_function(load_waveform, [path], tf.float32)
    audio.set_shape([AUDIO_LENGTH])

    spec = make_spectrogram(audio)
    spec.set_shape([
        (AUDIO_LENGTH - FRAME_LENGTH) // FRAME_STEP + 1,
        FFT_LENGTH // 2 + 1,
        1,
    ])
    return spec, label

def build_tf_dataset(split, training=False):
    subset = split_df[split_df["split"] == split]
    paths = subset["path"].values
    labels = subset["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training and len(paths) > 0:
        ds = ds.shuffle(
            len(paths),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(load_example, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_tf_dataset("train", training=True)
val_ds = build_tf_dataset("val")
test_ds = build_tf_dataset("test")

for sample_x, sample_y in train_ds.take(1):
    INPUT_SHAPE = tuple(sample_x.shape[1:])

print("Model input shape:", INPUT_SHAPE)

In [ ]:
# 7. Visualize an example spectrogram
sample_spec, sample_label = next(iter(train_ds))
plt.figure(figsize=(9, 4))
plt.imshow(
    sample_spec[0, :, :, 0].numpy().T,
    origin="lower",
    aspect="auto",
)
plt.title("Example Audio Spectrogram")
plt.xlabel("Time Frames")
plt.ylabel("Frequency Bins")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Build and compile the CNN classifier
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=INPUT_SHAPE),

    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# 9. Train the model
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/best_audio_classifier.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
]

print("Training CNN...")
start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1,
)

training_seconds = time.time() - start_time
print(f"Training time: {training_seconds:.2f} seconds")

In [ ]:
# 10. Plot training and validation curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150)
plt.show()

In [ ]:
# 11. Evaluate on the test set
y_true = []
y_pred = []

for x_batch, y_batch in test_ds:
    probabilities = model.predict(x_batch, verbose=0)
    y_pred.extend(np.argmax(probabilities, axis=1))
    y_true.extend(y_batch.numpy())

y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("========== TEST RESULTS ==========")
print(f"Test Accuracy  : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall   : {recall:.4f}")
print(f"Macro F1-score : {f1:.4f}")
print(f"Training Time  : {training_seconds:.2f} seconds")

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES,
    zero_division=0,
))

metrics_df = pd.DataFrame({
    "Metric": [
        "Test Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-score",
        "Training Time (seconds)",
    ],
    "Value": [accuracy, precision, recall, f1, training_seconds],
})
metrics_df.to_csv("/content/evaluation_results.csv", index=False)
display(metrics_df)

In [ ]:
# 12. Plot the confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title("Audio Classification Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# 13. Save the trained model
MODEL_PATH = "/content/audio_classification_model.keras"
model.save(MODEL_PATH)
print("Model saved:", MODEL_PATH)

In [ ]:
# 14. Speech-to-text (STT)
# SpeechRecognition's Google recognizer requires an internet connection.
def speech_to_text(audio_path, language="en-IN"):
    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(audio_path) as source:
            audio_data = recognizer.record(source)

        text = recognizer.recognize_google(audio_data, language=language)
        print("STT transcription:", text)
        return text

    except sr.UnknownValueError:
        print("STT could not understand the audio.")
        return ""
    except sr.RequestError as exc:
        print("STT service error:", exc)
        return ""

In [ ]:
# 15. Text-to-speech (TTS)
# gTTS requires an internet connection.
def text_to_speech(text, output_path="/content/speech_output.mp3"):
    if not text or not text.strip():
        print("No text to synthesize.")
        return None

    try:
        gTTS(text=text, lang="en", slow=False).save(output_path)
        print("TTS saved:", output_path)
        display(Audio(output_path))
        return output_path
    except Exception as exc:
        print("TTS error:", exc)
        return None

In [ ]:
# 16. Classify a WAV audio file
def classify_audio(audio_path):
    waveform = load_waveform(audio_path)
    spec = make_spectrogram(
        tf.convert_to_tensor(waveform, dtype=tf.float32)
    )
    spec = spec[tf.newaxis, ...]

    probabilities = model.predict(spec, verbose=0)[0]
    idx = int(np.argmax(probabilities))

    result = {
        "predicted_class": CLASS_NAMES[idx],
        "confidence": float(probabilities[idx]),
        "probabilities": {
            name: float(probabilities[i])
            for i, name in enumerate(CLASS_NAMES)
        },
    }

    print("\nPredicted command:", result["predicted_class"])
    print(f"Model confidence: {result['confidence']:.2%}")
    return result

In [ ]:
# 17. Upload audio and run the STT + CNN + TTS demo (Google Colab only)
from google.colab import files

print("Upload a WAV audio file to test STT and CNN classification.")
uploaded = files.upload()

if uploaded:
    test_audio_path = next(iter(uploaded.keys()))

    print("\n========== CNN CLASSIFICATION ==========")
    result = classify_audio(test_audio_path)

    print("\n========== SPEECH-TO-TEXT ==========")
    transcript = speech_to_text(test_audio_path)

    print("\n========== TEXT-TO-SPEECH ==========")
    response = f"The detected command is {result['predicted_class']}."
    text_to_speech(response)

    print("\n========== FINAL SUMMARY ==========")
    print("Transcription:", transcript or "Unavailable")
    print("Predicted class:", result["predicted_class"])
    print(f"Confidence: {result['confidence']:.2%}")
else:
    print("No file was uploaded.")

In [ ]:
# 18. List generated output files
output_paths = [
    "/content/audio_classification_model.keras",
    "/content/best_audio_classifier.keras",
    "/content/evaluation_results.csv",
    "/content/training_curves.png",
    "/content/confusion_matrix.png",
    "/content/speech_output.mp3",
]

print("Generated files:")
for path in output_paths:
    if os.path.exists(path):
        print(path)

print("\nPROJECT EXECUTION COMPLETED.")

In [ ]:
# Optional: download the evaluation report and plots from Google Colab
# Uncomment any line below to download that file.
# files.download("/content/evaluation_results.csv")
# files.download("/content/training_curves.png")
# files.download("/content/confusion_matrix.png")
# files.download("/content/audio_classification_model.keras")